# **Kalshi NBA Player Prop Market Analysis**

In [1]:
import numpy as np
import pandas as pd
from utils import *
import plotly.express as px
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing

In [2]:
SERIES_TICKERS = {
    "points"         : "KXNBAPTS",    # NBA: 'Duncan Robinson scores 10+ points in a game'
    "assists"        : "KXNBAAST",    # NBA: 'Duncan Robinson has 2+ assists in a game'
    "rebounds"       : "KXNBAREB",    # NBA: 'Duncan Robinson has 2+ rebounds in a game'
    "threes"         : "KXNBA3PT",    # NBA: 'Duncan Robinson makes 3+ three point shots in a game'
    "steals"         : "KXNBASTL",    # NBA: 'Duncan Robinson has 2+ steals in a game'
    "blocks"         : "KXNBABLK",    # NBA: 'Rudy Gobert has 1+ blocks in a game'
    "double_double"  : "KXNBA2D",     # NBA: 'Anthony Edwards has a double-double in a game'
    "triple_double"  : "KXNBA3D",     # NBA: 'Anthony Edwards has a triple-double in a game'
}

### Preprocessing - Market Data

In [3]:
# Load in data

recent_markets_df = pd.read_parquet('data/raw/recent_markets.parquet')
historical_markets_df = pd.read_parquet('data/raw/historical_markets.parquet')

In [4]:
recent_markets_df.head(2)

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2026-06-14T04:00:24Z,2026-06-11T04:00:34.852573Z,{'basketball_player': 'ee7a2d49-1c75-458b-992c...,This market will close and expire early if the...,KXNBAPTS-26JUN13NYKSAS,2026-06-14T03:30:00Z,2026-06-28T00:30:00Z,points,24.5,...,KXNBAPTS-26JUN13NYKSAS-NYKJHART3-25,Josh Hart: 25+ points,2026-06-14T04:06:23.732504Z,0.00,2223.94,1.0000,0.00,0.0000,0.00,Josh Hart: 25+
1,True,2026-06-14T04:00:24Z,2026-06-11T04:00:34.852573Z,{'basketball_player': 'ee7a2d49-1c75-458b-992c...,This market will close and expire early if the...,KXNBAPTS-26JUN13NYKSAS,2026-06-14T03:30:00Z,2026-06-28T00:30:00Z,points,19.5,...,KXNBAPTS-26JUN13NYKSAS-NYKJHART3-20,Josh Hart: 20+ points,2026-06-14T04:06:23.732504Z,0.00,21380.93,1.0000,0.00,0.0000,0.00,Josh Hart: 20+


In [5]:
historical_markets_df.head(2)

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2026-04-28T04:40:48Z,2026-04-27T04:01:46.3201Z,{'basketball_player': '07a246cd-1c40-42f1-849e...,This market will close and expire early if the...,KXNBAPTS-26APR27OKCPHX,2026-04-28T04:30:00Z,2026-05-12T01:30:00Z,points,19.5,...,KXNBAPTS-26APR27OKCPHX-PHXGALLEN8-20,Grayson Allen: 20+ points,2026-04-28T04:46:48.996477Z,0.00,2589.17,0.5600,-5.00,0.0000,0.00,Grayson Allen: 20+
1,True,2026-04-28T04:40:48Z,2026-04-27T04:01:41.12044Z,{'basketball_player': '07a246cd-1c40-42f1-849e...,This market will close and expire early if the...,KXNBAPTS-26APR27OKCPHX,2026-04-28T04:30:00Z,2026-05-12T01:30:00Z,points,14.5,...,KXNBAPTS-26APR27OKCPHX-PHXGALLEN8-15,Grayson Allen: 15+ points,2026-04-28T04:46:48.996477Z,0.00,3068.23,0.1600,-23.20,0.0000,0.00,Grayson Allen: 15+


In [6]:
# # Join data frames

markets_df = pd.concat([recent_markets_df, historical_markets_df], axis=0, join='outer')

In [7]:
# Drop duplicate tickers

markets_df = markets_df.drop_duplicates(['ticker'])

In [8]:
# Filter for liquid markets

markets_df = markets_df[markets_df['volume_fp'].astype(float) > 0]

In [9]:
# Market is finalized

markets_df = markets_df[markets_df['status'] == 'finalized']

In [10]:
# Remove 'scalar' (ie the game was cancelled)

markets_df = markets_df[markets_df['result'] != 'scalar']

In [11]:
# Clean up index

markets_df = markets_df.reset_index(drop=True)

In [12]:
# Create player column

split_col = markets_df['yes_sub_title'].str.split(':', n=1, expand=True)
markets_df['player']  = split_col[0].str.strip()

markets_df['player'].value_counts()[:10]

player
Victor Wembanyama          1633
Jalen Brunson              1547
James Harden               1496
LeBron James               1495
De'Aaron Fox               1493
Paolo Banchero             1403
Scottie Barnes             1376
LaMelo Ball                1373
Shai Gilgeous-Alexander    1373
Donovan Mitchell           1359
Name: count, dtype: int64

In [13]:
# Create series column

markets_df['series'] = markets_df['ticker'].apply(lambda x: x.split('-')[0])

### Preprocessing - Trade Data

In [14]:
# Load in data

recent_trades_df = pd.read_parquet('data/raw/recent_trades.parquet')
historical_trades_df = pd.read_parquet('data/raw/historical_trades.parquet')

In [15]:
recent_trades_df.head(2)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars
0,11.20,2026-06-14T02:47:44.568494Z,False,0.9900,ask,no,no,KXNBAPTS-26JUN13NYKSAS-NYKKTOWNS32-20,5f4eeabc-dfd7-4561-3d70-445385a0b201,0.0100
1,61.06,2026-06-14T02:41:09.201792Z,False,0.9900,ask,no,no,KXNBAPTS-26JUN13NYKSAS-NYKKTOWNS32-20,cab4ff55-8feb-46a9-bb9f-5d164d930d99,0.0100


In [16]:
recent_trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1414405 entries, 0 to 1414404
Data columns (total 10 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   count_fp            1414405 non-null  object
 1   created_time        1414405 non-null  object
 2   is_block_trade      1414405 non-null  bool  
 3   no_price_dollars    1414405 non-null  object
 4   taker_book_side     1414405 non-null  object
 5   taker_outcome_side  1414405 non-null  object
 6   taker_side          1414405 non-null  object
 7   ticker              1414405 non-null  object
 8   trade_id            1414405 non-null  object
 9   yes_price_dollars   1414405 non-null  object
dtypes: bool(1), object(9)
memory usage: 98.5+ MB


In [17]:
historical_trades_df.head(2)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars
0,22.85,2026-04-28T03:04:40.489615Z,False,0.0100,ask,no,no,KXNBAPTS-26APR27OKCPHX-PHXGALLEN8-10,99a64057-a8ea-64a3-9c30-4da46a50f416,0.9900
1,9.59,2026-04-28T03:02:48.473605Z,False,0.0100,ask,no,no,KXNBAPTS-26APR27OKCPHX-PHXGALLEN8-10,e1d2d332-9836-57f7-0d5d-13953874537a,0.9900


In [18]:
historical_trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1494901 entries, 0 to 1494900
Data columns (total 10 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   count_fp            1494901 non-null  object
 1   created_time        1494901 non-null  object
 2   is_block_trade      1494901 non-null  bool  
 3   no_price_dollars    1494901 non-null  object
 4   taker_book_side     1494901 non-null  object
 5   taker_outcome_side  1494901 non-null  object
 6   taker_side          1494901 non-null  object
 7   ticker              1494901 non-null  object
 8   trade_id            1494901 non-null  object
 9   yes_price_dollars   1494901 non-null  object
dtypes: bool(1), object(9)
memory usage: 104.1+ MB


In [19]:
# Join data frames

trades_df = pd.concat([recent_trades_df, historical_trades_df], axis=0, join='outer')

In [20]:
# Convert data

numeric_cols = ['count_fp', 'yes_price_dollars', 'no_price_dollars']

trades_df[numeric_cols] = trades_df[numeric_cols].apply(pd.to_numeric)

trades_df['created_time'] = pd.to_datetime(trades_df['created_time'], format='ISO8601', utc=True).dt.tz_convert('America/New_York')

In [21]:
# Drop duplicate trades

trades_df = trades_df.drop_duplicates(['trade_id']).reset_index(drop=True)

In [22]:
# Add dollar amount column

trades_df['dollar_amt'] = trades_df['count_fp'] * np.where(
    trades_df['taker_outcome_side'] == 'no', 
    trades_df['no_price_dollars'],
    trades_df['yes_price_dollars']
)

In [23]:
# Create game date, away team, and home team columns

date_teams = trades_df['ticker'].str.split('-', n=1).str[1]

year  = date_teams.str[0:2]      # "26"
month = date_teams.str[2:5]      # "JUN"
day   = date_teams.str[5:7]      # "13"

trades_df['game_date'] = pd.to_datetime(
    year + month + day,
    format='%y%b%d'
)

trades_df['away_team'] = date_teams.str[7:10]
trades_df['home_team'] = date_teams.str[10:13]

trades_df.head(3)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars,dollar_amt,game_date,away_team,home_team
0,11.20,2026-06-13 22:47:44.568494-04:00,False,0.99,ask,no,no,KXNBAPTS-26JUN13NYKSAS-NYKKTOWNS32-20,5f4eeabc-dfd7-4561-3d70-445385a0b201,0.01,11.0880,2026-06-13,NYK,SAS
1,61.06,2026-06-13 22:41:09.201792-04:00,False,0.99,ask,no,no,KXNBAPTS-26JUN13NYKSAS-NYKKTOWNS32-20,cab4ff55-8feb-46a9-bb9f-5d164d930d99,0.01,60.4494,2026-06-13,NYK,SAS
2,87.48,2026-06-13 22:40:03.490660-04:00,False,0.98,ask,no,no,KXNBAPTS-26JUN13NYKSAS-NYKKTOWNS32-20,d4e9265b-85b9-53d3-e15c-1335e5d4a4d4,0.02,85.7304,2026-06-13,NYK,SAS


### Preprocessing - NBA Game Start Data

In [24]:
nba_schedule_df = pd.read_parquet('data/raw/nba_start_times.parquet')

In [25]:
# Clean up data frame

nba_schedule_df = (
    nba_schedule_df[
        ["gameDate", "gameDateTimeEst", "awayTeam_teamTricode", "homeTeam_teamTricode"]
    ]
    .astype({
        "gameDate": "datetime64[ns, UTC]",
        "gameDateTimeEst": "datetime64[ns, UTC]",
        "awayTeam_teamTricode": "str",
        "homeTeam_teamTricode": "str",
    })
    .rename(columns={
        "gameDate": "game_date",
        "gameDateTimeEst": "game_start_time",
        "awayTeam_teamTricode": "away_team",
        "homeTeam_teamTricode": "home_team",
    })
)

nba_schedule_df["game_date"] = nba_schedule_df["game_date"].dt.tz_localize(None)

nba_schedule_df

,game_date,game_start_time,away_team,home_team
0,2024-10-04,2024-10-04 12:00:00+00:00,BOS,DEN
1,2024-10-04,2024-10-04 21:00:00+00:00,NZB,UTA
2,2024-10-04,2024-10-04 22:30:00+00:00,MIN,LAL
3,2024-10-05,2024-10-05 19:00:00+00:00,GSW,LAC
4,2024-10-06,2024-10-06 10:00:00+00:00,DEN,BOS
...,...,...,...,...
2795,2026-06-03,2026-06-03 20:30:00+00:00,NYK,SAS
2796,2026-06-05,2026-06-05 20:30:00+00:00,NYK,SAS
2797,2026-06-08,2026-06-08 20:30:00+00:00,SAS,NYK
2798,2026-06-10,2026-06-10 20:30:00+00:00,SAS,NYK


In [26]:
# Add game start time column to trades data frame

trades_df = trades_df.merge(
    nba_schedule_df,
    on=['game_date', 'away_team', 'home_team'],
    how='left'
)

In [27]:
# Data validation

print(f"Missing game start time for {trades_df['game_start_time'].isna().sum():,} out of {len(trades_df['game_start_time']):,} trades.")

Missing game start time for 1,475 out of 2,720,134 trades.


### Preprocessing - Create Prop Data Frames

In [28]:
markets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80193 entries, 0 to 80192
Data columns (total 49 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   can_close_early             80193 non-null  bool   
 1   close_time                  80193 non-null  object 
 2   created_time                80193 non-null  object 
 3   custom_strike               80193 non-null  object 
 4   early_close_condition       80193 non-null  object 
 5   event_ticker                80193 non-null  object 
 6   expected_expiration_time    80193 non-null  object 
 7   expiration_time             80193 non-null  object 
 8   expiration_value            80193 non-null  object 
 9   floor_strike                74744 non-null  float64
 10  fractional_trading_enabled  80193 non-null  bool   
 11  last_price_dollars          80193 non-null  object 
 12  latest_expiration_time      80193 non-null  object 
 13  liquidity_dollars           801

In [29]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2720134 entries, 0 to 2720133
Data columns (total 15 columns):
 #   Column              Dtype                           
---  ------              -----                           
 0   count_fp            float64                         
 1   created_time        datetime64[ns, America/New_York]
 2   is_block_trade      bool                            
 3   no_price_dollars    float64                         
 4   taker_book_side     object                          
 5   taker_outcome_side  object                          
 6   taker_side          object                          
 7   ticker              object                          
 8   trade_id            object                          
 9   yes_price_dollars   float64                         
 10  dollar_amt          float64                         
 11  game_date           datetime64[ns]                  
 12  away_team           object                          
 13  home_team   

In [30]:
# Merge data frames

merged = trades_df.merge(markets_df, on='ticker', how='left')

final = merged[[
    'ticker', 'created_time_x', 'count_fp', 'close_time', 'series',
    'game_start_time', 'dollar_amt', 'result', 'yes_price_dollars',
    'player',
]].rename(columns={'created_time_x': 'created_time'}).dropna().reset_index(drop=True).copy()

final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2688587 entries, 0 to 2688586
Data columns (total 10 columns):
 #   Column             Dtype                           
---  ------             -----                           
 0   ticker             object                          
 1   created_time       datetime64[ns, America/New_York]
 2   count_fp           float64                         
 3   close_time         object                          
 4   series             object                          
 5   game_start_time    datetime64[ns, UTC]             
 6   dollar_amt         float64                         
 7   result             object                          
 8   yes_price_dollars  float64                         
 9   player             object                          
dtypes: datetime64[ns, America/New_York](1), datetime64[ns, UTC](1), float64(3), object(5)
memory usage: 205.1+ MB


In [31]:
# Create 'prop' data frames

prop_dfs = {
    name: final[final['series'] == ticker].reset_index(drop=True).copy()
    for name, ticker in SERIES_TICKERS.items()
}

double_double_df = prop_dfs['double_double']
triple_double_df = prop_dfs['triple_double']
threes_df        = prop_dfs['threes']
steals_df        = prop_dfs['steals']
blocks_df        = prop_dfs['blocks']
points_df        = prop_dfs['points']
assists_df       = prop_dfs['assists']
rebounds_df      = prop_dfs['rebounds']

In [32]:
print('Dollar Volume by Series:')
final.groupby('series')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Dollar Volume by Series:


series
KXNBA2D      $2,760,483
KXNBA3D        $969,556
KXNBA3PT    $10,611,468
KXNBAAST     $7,970,877
KXNBABLK     $1,036,954
KXNBAPTS    $50,642,614
KXNBAREB    $11,844,190
KXNBASTL     $1,379,505
Name: dollar_amt, dtype: object

In [33]:
print('Number of Markets (tickers) by Series:')
final.groupby('series')['ticker'].count().apply(lambda x: f"{x:,}")

Number of Markets (tickers) by Series:


series
KXNBA2D       159,616
KXNBA3D        63,146
KXNBA3PT      334,855
KXNBAAST      229,194
KXNBABLK       37,538
KXNBAPTS    1,501,605
KXNBAREB      316,906
KXNBASTL       45,727
Name: ticker, dtype: object

In [34]:
print('First and Last Trade Dates by Series:')
final.groupby('series').agg(
    first_trade_date = ('created_time', 'min'),
    last_trade_date  = ('created_time', 'max')
)

First and Last Trade Dates by Series:


,first_trade_date,last_trade_date
series,,
KXNBA2D,2025-12-03 12:39:20.700353-05:00,2026-06-13 23:26:40.728336-04:00
KXNBA3D,2025-12-03 14:12:42.670903-05:00,2026-06-13 23:09:40.071990-04:00
KXNBA3PT,2025-11-18 07:35:47.680422-05:00,2026-06-13 23:30:47.124992-04:00
KXNBAAST,2025-11-18 07:39:55.882969-05:00,2026-06-13 23:27:00.595040-04:00
KXNBABLK,2026-01-13 20:25:25.964181-05:00,2026-06-13 23:30:32.344054-04:00
KXNBAPTS,2025-11-18 02:59:29.981212-05:00,2026-06-13 23:30:40.500813-04:00
KXNBAREB,2025-11-18 12:50:58.539217-05:00,2026-06-13 23:30:31.528444-04:00
KXNBASTL,2026-01-13 20:13:53.164780-05:00,2026-06-13 23:29:33.973801-04:00


# Analysis

In [35]:
LIQUIDITY_AMOUNT: float = 25.0      # Pre-game liquidity size for simulation
TRADES: int = 5                     # Required number of pre-game trades for simulation
DOLLAR_VOLUME: float = 50.0         # Required dollar volume pre-game for simulation
KALSHI_MAKER_FEE: float = 0.0175    # Kalshi maker fee for June 2026

### Analysis - **Double Double** in a NBA Game

In [36]:
# Aggregate probabilities

pregame_prob, hit_rate, n = agg_probabilities(
    df=double_double_df, 
    trades=TRADES,                  # Filter for strictly greater than x number of trades pregame, and
    dollar_volume=DOLLAR_VOLUME     # filter for strictly greater than y dollar volume pregame
)

if n is not None:
    print(f"Total trades: {n:,}")
    print(f"Pre-game market probability of a double double: {pregame_prob*100:.2f}%")
    print(f"Hit-rate of a double double: {hit_rate*100:.2f}%")

Total trades: 29,120
Pre-game market probability of a double double: 38.45%
Hit-rate of a double double: 24.30%


In [37]:
# Player hit rates

players = player_hit_rates(
    df=double_double_df, 
    n_prop_markets=30,              # Large number of observed markets       
    trades=TRADES,
    dollar_volume=DOLLAR_VOLUME
)

players.sort_values(by='hit_rate_dvwa_delta', ascending=False)

,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Karl-Anthony Towns,37.0,1047.0,19835.39,0.70,0.66,0.65,0.04,0.05
Stephon Castle,39.0,1289.0,21103.66,0.26,0.22,0.21,0.04,0.05
Josh Hart,32.0,1282.0,24593.90,0.22,0.25,0.25,-0.03,-0.03
Luka Dončić,32.0,686.0,12935.49,0.50,0.54,0.52,-0.04,-0.02
Chet Holmgren,32.0,817.0,15690.88,0.34,0.40,0.38,-0.06,-0.04
Rudy Gobert,44.0,881.0,27975.18,0.39,0.45,0.42,-0.07,-0.03
Austin Reaves,32.0,637.0,8314.93,0.00,0.10,0.08,-0.10,-0.08
James Harden,31.0,580.0,9806.06,0.16,0.28,0.21,-0.12,-0.05
Shai Gilgeous-Alexander,35.0,910.0,10166.61,0.11,0.27,0.24,-0.15,-0.13


In [38]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every double double prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

sim_results = simulation(
    df=double_double_df,
    side='no',                      # Selling 'yes' is the same as buying 'no'
    liquidity_amt=LIQUIDITY_AMOUNT,
    fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
    by='dvwa',
    trades=TRADES,                  # Strictly more than x trades occured pre-game
    dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
)

fig = px.line(sim_results, x='close_time', y='cum_net_winnings')

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Double Double Prop Simulation | {len(sim_results):,} Markets | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.show()

In [39]:
# Simulation Statistics: Sell Yes

w_simulation_stats = simulation_statistics(
    df=sim_results, 
    group_returns_by='weekly'
)
m_simulation_stats = simulation_statistics(
    df=sim_results, 
    group_returns_by='monthly'
)

print('Weekly Returns:', w_simulation_stats)
print('Monthly Returns:', m_simulation_stats)

Weekly Returns: {'median': 0.12041847791678251, 'mean': 0.19778853842147892, 'std': 0.41516029225056206, 'count': 27}
Monthly Returns: {'median': 0.12709001240838666, 'mean': 0.11662793096906308, 'std': 0.10485920201881982, 'count': 7}


### Analysis - **Triple Double** in a NBA Game

In [40]:
# Aggregate probabilities

pregame_prob, hit_rate, n = agg_probabilities(
    df=triple_double_df, 
    trades=TRADES, 
    dollar_volume=DOLLAR_VOLUME
)

if n is not None:
    print(f"Total trades: {n:,}")
    print(f"Pre-game market probability of a triple double: {pregame_prob*100:.2f}%")
    print(f"Hit-rate of a triple double: {hit_rate*100:.2f}%")

Total trades: 13,443
Pre-game market probability of a triple double: 16.43%
Hit-rate of a triple double: 8.35%


In [41]:
# Player hit rates

players = player_hit_rates(
    df=triple_double_df, 
    n_prop_markets=30,              # Large number of observed markets
    trades=TRADES,
    dollar_volume=DOLLAR_VOLUME
)

players.sort_values(by='hit_rate_dvwa_delta', ascending=False)

,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,36.0,2914.0,38818.76,0.03,0.04,0.03,-0.01,-0.00
Josh Hart,34.0,1034.0,10811.30,0.03,0.05,0.04,-0.02,-0.01
LeBron James,48.0,1717.0,17337.80,0.06,0.09,0.07,-0.02,-0.01
Nikola Jokić,37.0,1301.0,34447.93,0.51,0.54,0.54,-0.03,-0.03
Cade Cunningham,31.0,723.0,7508.55,0.00,0.08,0.07,-0.08,-0.07
Luka Dončić,36.0,1071.0,16773.08,0.08,0.16,0.15,-0.08,-0.07


In [42]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every triple double prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

sim_results = simulation(
    df=triple_double_df,
    side='no',                      # Selling 'yes' is the same as buying 'no'
    liquidity_amt=LIQUIDITY_AMOUNT,
    fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
    by='dvwa',
    trades=TRADES,                  # Strictly more than x trades occured pre-game
    dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
)

fig = px.line(sim_results, x='close_time', y='cum_net_winnings')

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Triple Double Prop Simulation | {len(sim_results):,} Markets | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.show()

In [43]:
# Simulation Statistics: Sell Yes

w_simulation_stats = simulation_statistics(
    df=sim_results, 
    group_returns_by='weekly'
)
m_simulation_stats = simulation_statistics(
    df=sim_results, 
    group_returns_by='monthly'
)

print('Weekly Returns:', w_simulation_stats)
print('Monthly Returns:', m_simulation_stats)

Weekly Returns: {'median': 0.04539033861529087, 'mean': 0.046394710794190386, 'std': 0.09568400483744498, 'count': 27}
Monthly Returns: {'median': 0.05525103936803532, 'mean': 0.04809352836639104, 'std': 0.028521833275241052, 'count': 7}


### Analysis - **Threes** in a NBA Game

In [44]:
# Add number of threes column

threes_df['n_threes'] = threes_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

three_frequencies = Counter(threes_df['n_threes'])

three_frequencies

Counter({2: 106463, 3: 102955, 4: 49315, 1: 44603, 5: 27640, 6: 3419, 7: 460})

In [45]:
# Total dollar volume by n threes market

print("Total dollar volume by n threes market:")
threes_df.groupby('n_threes')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n threes market:


n_threes
1    $2,084,035
2    $3,564,625
3    $2,962,729
4    $1,325,561
5      $577,075
6       $88,158
7        $9,286
Name: dollar_amt, dtype: object

In [46]:
# Aggregate probabilities

results = []

for threes, freq in sorted(three_frequencies.items()):
    df = threes_df[threes_df['n_threes'] == threes].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 10, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'threes'       : f"{threes}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['threes'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['threes'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['threes'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Threes Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Three Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [47]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['threes']} threes ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ threes ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
LeBron James,17.0,253.0,7174.89,0.82,0.76,0.78,0.06,0.04
Josh Hart,13.0,472.0,21671.30,0.77,0.72,0.81,0.05,-0.04
Stephon Castle,26.0,578.0,25988.56,0.77,0.74,0.76,0.03,0.00
Karl-Anthony Towns,17.0,330.0,12749.63,0.76,0.74,0.76,0.02,0.00
Victor Wembanyama,17.0,890.0,14704.36,0.82,0.88,0.89,-0.05,-0.07
Shai Gilgeous-Alexander,18.0,297.0,22556.88,0.72,0.81,0.81,-0.09,-0.09
De'Aaron Fox,13.0,228.0,4965.76,0.62,0.78,0.77,-0.17,-0.15
Chet Holmgren,17.0,256.0,7963.41,0.47,0.70,0.74,-0.23,-0.27



──────────── 2+ threes ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
OG Anunoby,19.0,464.0,11328.82,0.84,0.67,0.66,0.17,0.18
Jalen Johnson,12.0,160.0,6939.54,0.67,0.52,0.50,0.15,0.17
Kevin Durant,18.0,178.0,6041.07,0.83,0.70,0.70,0.13,0.13
Duncan Robinson,11.0,161.0,2919.62,0.82,0.70,0.68,0.11,0.14
Desmond Bane,15.0,188.0,5897.96,0.73,0.65,0.64,0.09,0.09
Victor Wembanyama,35.0,2196.0,53431.89,0.66,0.60,0.61,0.06,0.05
Cade Cunningham,23.0,386.0,26742.37,0.74,0.68,0.66,0.06,0.08
Devin Booker,15.0,245.0,5661.02,0.67,0.62,0.60,0.05,0.07
Cooper Flagg,13.0,173.0,4076.60,0.38,0.35,0.35,0.03,0.03



──────────── 3+ threes ────────────


...


──────────── 4+ threes ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Luka Dončić,22.0,435.0,18898.11,0.68,0.58,0.57,0.10,0.11
Jalen Brunson,15.0,393.0,7225.59,0.27,0.25,0.24,0.02,0.03
LaMelo Ball,26.0,391.0,10102.39,0.58,0.55,0.55,0.02,0.03
Victor Wembanyama,23.0,699.0,10485.86,0.13,0.16,0.16,-0.03,-0.03
Anthony Edwards,23.0,328.0,8668.08,0.35,0.41,0.42,-0.06,-0.07
Kon Knueppel,26.0,483.0,9860.29,0.38,0.46,0.47,-0.08,-0.09
Stephon Castle,13.0,287.0,2163.72,0.00,0.09,0.09,-0.09,-0.09
Tyrese Maxey,13.0,150.0,2553.16,0.23,0.37,0.36,-0.13,-0.13
Donovan Mitchell,14.0,154.0,3103.65,0.21,0.39,0.37,-0.18,-0.16



──────────── 5+ threes ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,15.0,302.0,4437.77,0.07,0.08,0.07,-0.01,-0.0



──────────── 6+ threes ────────────
No data available for given constraint.

──────────── 7+ threes ────────────
No data available for given constraint.


In [48]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every threes prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_threes = set(int(r['threes'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_threes_map = dict()
for n in n_threes:
    df = threes_df[threes_df['n_threes'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_threes'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_threes_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_threes_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_threes'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Threes Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Threes'
)

fig.show()

In [49]:
# Simulation Statistics: Sell Yes, by Threes Threshold

stats_rows = []
for n, data in n_threes_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_threes': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_threes')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_threes'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_threes'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_threes'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Threes Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Threes', row=1, col=1)
fig.update_xaxes(title_text='n+ Threes', row=1, col=2)
fig.update_xaxes(title_text='n+ Threes', row=2, col=1)
fig.update_xaxes(title_text='n+ Threes', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Steals** in a NBA Game

In [50]:
# Add number of steals column

steals_df['n_steals'] = steals_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

steal_frequencies = Counter(steals_df['n_steals'])

steal_frequencies

Counter({2: 19522, 1: 17991, 3: 8205, 4: 9})

In [51]:
# Total dollar volume by n steals market

print("Total dollar volume by n steals market:")
steals_df.groupby('n_steals')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n steals market:


n_steals
1    $711,781
2    $518,088
3    $148,979
4        $657
Name: dollar_amt, dtype: object

In [52]:
# Aggregate probabilities

results = []

for steals, freq in sorted(steal_frequencies.items()):
    df = steals_df[steals_df['n_steals'] == steals].copy()
    
    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 5, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'steals'       : f"{steals}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['steals'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['steals'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['steals'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Steals Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Steal Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [53]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['steals']} steals ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ steals ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Stephon Castle,11.0,195.0,4127.93,0.82,0.66,0.67,0.15,0.15
Josh Hart,9.0,492.0,5691.50,0.78,0.69,0.69,0.09,0.09
Victor Wembanyama,11.0,356.0,5200.62,0.73,0.67,0.72,0.05,0.01
Mikal Bridges,7.0,174.0,3068.75,0.71,0.68,0.69,0.04,0.02
OG Anunoby,7.0,121.0,1534.59,0.71,0.68,0.73,0.03,-0.02
Devin Vassell,13.0,214.0,4395.73,0.62,0.64,0.64,-0.02,-0.02
Jaden McDaniels,7.0,85.0,1274.01,0.43,0.61,0.63,-0.18,-0.20
Jalen Brunson,8.0,192.0,4987.63,0.25,0.61,0.61,-0.36,-0.36



──────────── 2+ steals ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Devin Vassell,7.0,190.0,2083.81,0.43,0.31,0.33,0.12,0.10
Shai Gilgeous-Alexander,12.0,157.0,2214.63,0.50,0.40,0.43,0.10,0.07
Dyson Daniels,6.0,68.0,1316.10,0.67,0.59,0.60,0.08,0.06
Jalen Brunson,6.0,155.0,2570.93,0.33,0.27,0.26,0.06,0.07
Josh Hart,14.0,523.0,10852.13,0.43,0.40,0.35,0.03,0.08
Marcus Smart,8.0,124.0,2812.59,0.50,0.49,0.48,0.01,0.02
Tyrese Maxey,8.0,98.0,1160.12,0.38,0.37,0.37,0.00,0.01
OG Anunoby,8.0,110.0,1177.44,0.38,0.39,0.39,-0.02,-0.02
Victor Wembanyama,12.0,249.0,2623.02,0.33,0.36,0.36,-0.03,-0.03



──────────── 3+ steals ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Josh Hart,7.0,173.0,1461.51,0.29,0.17,0.17,0.12,0.12
Devin Vassell,7.0,131.0,944.98,0.14,0.13,0.13,0.01,0.01
Stephon Castle,7.0,155.0,819.90,0.14,0.13,0.13,0.01,0.01


In [54]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every steals prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_steals = set(int(r['steals'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_steals_map = dict()
for n in n_steals:
    df = steals_df[steals_df['n_steals'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_steals'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_steals_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_steals_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_steals'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Steals Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Steals'
)

fig.show()

In [55]:
# Simulation Statistics: Sell Yes, by Steals Threshold

stats_rows = []
for n, data in n_steals_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_steals': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_steals')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_steals'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_steals'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_steals'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Steals Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Steals', row=1, col=1)
fig.update_xaxes(title_text='n+ Steals', row=1, col=2)
fig.update_xaxes(title_text='n+ Steals', row=2, col=1)
fig.update_xaxes(title_text='n+ Steals', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Blocks** in a NBA Game

In [56]:
# Add number of blocks column

blocks_df['n_blocks'] = blocks_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

block_frequencies = Counter(blocks_df['n_blocks'])

block_frequencies

Counter({1: 10754,
         2: 9581,
         3: 7613,
         4: 5042,
         5: 4039,
         6: 492,
         12: 11,
         10: 5,
         14: 1})

In [57]:
# Total dollar volume by n blocks market

print("Total dollar volume by n blocks market:")
blocks_df.groupby('n_blocks')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n blocks market:


n_blocks
1     $415,618
2     $247,377
3     $172,871
4     $120,273
5      $66,852
6      $12,818
10        $510
12        $535
14         $99
Name: dollar_amt, dtype: object

In [58]:
# Aggregate probabilities

results = []

for blocks, freq in sorted(block_frequencies.items()):
    df = blocks_df[blocks_df['n_blocks'] == blocks].copy()
    
    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 10, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'blocks'       : f"{blocks}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['blocks'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['blocks'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['blocks'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Actual Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Blocks Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Block Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [59]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['blocks']} blocks ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ blocks ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Karl-Anthony Towns,15.0,676.0,18234.09,0.8,0.58,0.58,0.22,0.22



──────────── 2+ blocks ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Chet Holmgren,16.0,271.0,8168.84,0.44,0.51,0.55,-0.07,-0.11



──────────── 3+ blocks ────────────


...


──────────── 4+ blocks ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,28.0,1338.0,22502.85,0.32,0.5,0.47,-0.18,-0.15



──────────── 5+ blocks ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,21.0,1170.0,13979.38,0.19,0.38,0.29,-0.19,-0.1



──────────── 6+ blocks ────────────
No data available for given constraint.


In [60]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every blocks prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_blocks = set(int(r['blocks'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_blocks_map = dict()
for n in n_blocks:
    df = blocks_df[blocks_df['n_blocks'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_blocks'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_blocks_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_blocks_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_blocks'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Blocks Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Blocks'
)

fig.show()

In [61]:
# Simulation Statistics: Sell Yes, by Blocks Threshold

stats_rows = []
for n, data in n_blocks_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_blocks': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_blocks')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_blocks'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_blocks'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_blocks'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Blocks Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Blocks', row=1, col=1)
fig.update_xaxes(title_text='n+ Blocks', row=1, col=2)
fig.update_xaxes(title_text='n+ Blocks', row=2, col=1)
fig.update_xaxes(title_text='n+ Blocks', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Points** in a NBA Game

In [62]:
# Add number of points column

points_df['n_points'] = points_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

point_frequencies = Counter(points_df['n_points'])

point_frequencies

Counter({20: 354690,
         25: 327180,
         15: 307995,
         10: 205402,
         30: 204145,
         35: 71266,
         40: 20891,
         13: 1384,
         45: 1152,
         19: 816,
         24: 788,
         18: 658,
         29: 630,
         28: 618,
         17: 582,
         23: 573,
         22: 523,
         21: 461,
         27: 369,
         32: 291,
         26: 269,
         31: 247,
         11: 145,
         33: 142,
         34: 118,
         16: 93,
         12: 91,
         14: 51,
         36: 15,
         50: 9,
         80: 6,
         82: 5})

In [63]:
# Total dollar volume by n points market

print("Total dollar volume by n points market:")
points_df.groupby('n_points')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n points market:


n_points
10     $7,151,154
11         $4,763
12         $3,480
13        $39,241
14         $2,228
15    $11,154,660
16         $4,307
17        $19,549
18        $29,730
19        $30,073
20    $12,218,668
21        $25,056
22        $25,965
23        $21,565
24        $26,772
25    $11,257,726
26        $16,820
27        $24,743
28        $19,869
29        $22,264
30     $6,242,068
31        $13,305
32         $9,514
33         $4,592
34         $2,584
35     $1,816,236
36           $788
40       $433,019
45        $21,023
50            $62
80           $495
82           $297
Name: dollar_amt, dtype: object

In [64]:
# Aggregate probabilities

results = []

for points, freq in sorted(point_frequencies.items()):
    if freq < 200:
        continue
    
    df = points_df[points_df['n_points'] == points].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 30, TRADES, DOLLAR_VOLUME) # Large number of observed markets

    results.append({
        'points'       : f"{points}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['points'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['points'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['points'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Actual Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Points Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Point Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [65]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['points']} points ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 10+ points ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Dylan Harper,35.0,3098.0,107546.70,0.71,0.62,0.60,0.09,0.11
Deandre Ayton,31.0,614.0,18717.77,0.52,0.59,0.54,-0.07,-0.02



──────────── 13+ points ────────────
No data available for given constraint.

──────────── 15+ points ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Stephon Castle,46.0,3046.0,114195.69,0.74,0.65,0.64,0.09,0.10
OG Anunoby,32.0,1705.0,75099.66,0.69,0.61,0.61,0.07,0.08
James Harden,35.0,1221.0,68609.30,0.83,0.77,0.78,0.06,0.05
De'Aaron Fox,44.0,2148.0,92087.38,0.59,0.55,0.55,0.04,0.04
Karl-Anthony Towns,31.0,1632.0,55222.49,0.71,0.70,0.68,0.01,0.03
Chet Holmgren,33.0,1390.0,46511.73,0.55,0.55,0.55,0.00,-0.00
Dylan Harper,34.0,3470.0,109384.46,0.41,0.43,0.35,-0.01,0.06
Mikal Bridges,40.0,2199.0,75337.69,0.40,0.43,0.44,-0.03,-0.04
VJ Edgecombe,37.0,1276.0,46530.27,0.43,0.48,0.43,-0.05,0.00



──────────── 17+ points ────────────
No data available for given constraint.

──────────── 18+ points ────────────
No data available for given constraint.

──────────── 19+ points ────────────
No data available for given constraint.

──────────── 20+ points ────────────


...


──────────── 27+ points ────────────
No data available for given constraint.

──────────── 28+ points ────────────
No data available for given constraint.

──────────── 29+ points ────────────
No data available for given constraint.

──────────── 30+ points ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Anthony Edwards,31.0,1336.0,41827.69,0.55,0.44,0.41,0.11,0.14
Luka Dončić,33.0,1396.0,58233.51,0.70,0.65,0.64,0.05,0.06
Jaylen Brown,34.0,514.0,12048.25,0.44,0.44,0.42,-0.00,0.02
Donovan Mitchell,42.0,1571.0,38950.40,0.38,0.40,0.39,-0.01,-0.01
Shai Gilgeous-Alexander,52.0,3181.0,127366.32,0.54,0.54,0.54,-0.01,-0.00
Jalen Brunson,47.0,2899.0,91360.38,0.32,0.38,0.39,-0.06,-0.07
Tyrese Maxey,37.0,984.0,34737.53,0.27,0.40,0.40,-0.13,-0.13
Victor Wembanyama,41.0,5500.0,180644.80,0.24,0.38,0.39,-0.14,-0.15
Nikola Jokić,31.0,915.0,25636.35,0.26,0.50,0.47,-0.25,-0.21



──────────── 31+ points ────────────
No data available for given constraint.

──────────── 32+ points ────────────
No data available for given constraint.

──────────── 35+ points ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shai Gilgeous-Alexander,31.0,773.0,19862.03,0.29,0.31,0.31,-0.02,-0.02
Victor Wembanyama,31.0,2078.0,52710.67,0.16,0.20,0.19,-0.04,-0.03



──────────── 40+ points ────────────
No data available for given constraint.

──────────── 45+ points ────────────
No data available for given constraint.


In [66]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every points prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_points = set(int(r['points'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_points_map = dict()
for n in n_points:
    df = points_df[points_df['n_points'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_points'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_points_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_points_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_points'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Points Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Points'
)

fig.show()

In [67]:
# Simulation Statistics: Sell Yes, by Points Threshold

stats_rows = []
for n, data in n_points_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_points': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_points')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_points'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_points'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_points'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Points Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Points', row=1, col=1)
fig.update_xaxes(title_text='n+ Points', row=1, col=2)
fig.update_xaxes(title_text='n+ Points', row=2, col=1)
fig.update_xaxes(title_text='n+ Points', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Assists** in a NBA Game

In [68]:
# Add number of assists column

assists_df['n_assists'] = assists_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

assist_frequencies = Counter(assists_df['n_assists'])

assist_frequencies

Counter({6: 55857,
         4: 43995,
         8: 38468,
         2: 22367,
         10: 20968,
         7: 16263,
         5: 11812,
         3: 7679,
         12: 6540,
         9: 2023,
         11: 1284,
         14: 1106,
         1: 749,
         16: 83})

In [69]:
# Total dollar volume by n assists market

print("Total dollar volume by n assists market:")
assists_df.groupby('n_assists')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n assists market:


n_assists
1        $22,127
2     $1,060,794
3       $253,781
4     $1,512,770
5       $421,293
6     $1,938,442
7       $601,667
8     $1,210,773
9        $84,269
10      $638,395
11       $39,102
12      $158,079
14       $27,072
16        $2,312
Name: dollar_amt, dtype: object

In [70]:
# Aggregate probabilities

results = []

for assists, freq in sorted(assist_frequencies.items()):
    df = assists_df[assists_df['n_assists'] == assists].copy()
    
    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 15, TRADES, DOLLAR_VOLUME) # Medium number of observed markets

    results.append({
        'assists'       : f"{assists}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['assists'],
    y=results_df['pregame_prob'],
    name='Pre-Game Market',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['assists'],
    y=results_df['hit_rate'],
    name='Actual Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['assists'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Assists Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Assist Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [71]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['assists']} assists ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ assists ────────────
No data available for given constraint.

──────────── 2+ assists ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,18.0,445.0,4589.62,0.83,0.86,0.85,-0.02,-0.02



──────────── 3+ assists ────────────
No data available for given constraint.

──────────── 4+ assists ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,29.0,848.0,17884.93,0.38,0.43,0.42,-0.05,-0.04



──────────── 5+ assists ────────────
No data available for given constraint.

──────────── 6+ assists ────────────


...


──────────── 9+ assists ────────────
No data available for given constraint.

──────────── 10+ assists ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
LeBron James,16.0,319.0,5657.47,0.50,0.45,0.41,0.05,0.09
Nikola Jokić,28.0,718.0,20265.20,0.54,0.56,0.57,-0.02,-0.03
Cade Cunningham,24.0,385.0,11463.97,0.38,0.54,0.53,-0.16,-0.16



──────────── 11+ assists ────────────
No data available for given constraint.

──────────── 12+ assists ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Nikola Jokić,23.0,409.0,9922.31,0.39,0.36,0.4,0.03,-0.01



──────────── 14+ assists ────────────
No data available for given constraint.


In [72]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every assists prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_assists = set(int(r['assists'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_assists_map = dict()
for n in n_assists:
    df = assists_df[assists_df['n_assists'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_assists'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_assists_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_assists_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_assists'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Assists Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Assists'
)

fig.show()

In [73]:
# Simulation Statistics: Sell Yes, by Assists Threshold

stats_rows = []
for n, data in n_assists_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_assists': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_assists')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_assists'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_assists'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_assists'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Assists Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Assists', row=1, col=1)
fig.update_xaxes(title_text='n+ Assists', row=1, col=2)
fig.update_xaxes(title_text='n+ Assists', row=2, col=1)
fig.update_xaxes(title_text='n+ Assists', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Rebounds** in a NBA Game

In [74]:
# Add number of rebounds column

rebounds_df['n_rebounds'] = rebounds_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

rebound_frequencies = Counter(rebounds_df['n_rebounds'])

rebound_frequencies

Counter({6: 60738,
         8: 58182,
         10: 45904,
         4: 39009,
         12: 32291,
         2: 14784,
         14: 13745,
         5: 13422,
         7: 11484,
         9: 8992,
         16: 6601,
         11: 3961,
         13: 3445,
         3: 2294,
         18: 1078,
         15: 810,
         20: 166})

In [75]:
# Total dollar volume by n rebounds market

print("Total dollar volume by n rebounds market:")
rebounds_df.groupby('n_rebounds')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n rebounds market:


n_rebounds
2       $873,171
3        $82,529
4     $1,625,091
5       $505,138
6     $2,372,345
7       $475,178
8     $2,109,049
9       $348,517
10    $1,472,137
11      $166,921
12    $1,081,197
13      $113,993
14      $406,753
15       $42,769
16      $152,417
18       $15,027
20        $1,959
Name: dollar_amt, dtype: object

In [76]:
# Aggregate probabilities

results = []

for rebounds, freq in sorted(rebound_frequencies.items()):
    df = rebounds_df[rebounds_df['n_rebounds'] == rebounds].copy()
    
    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 10, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'rebounds'       : f"{rebounds}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['rebounds'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['rebounds'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['rebounds'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Rebounds Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Rebound Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [77]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['rebounds']} rebounds ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 2+ rebounds ────────────
No data available for given constraint.

──────────── 3+ rebounds ────────────
No data available for given constraint.

──────────── 4+ rebounds ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Donovan Mitchell,14.0,203.0,5358.25,0.79,0.64,0.64,0.15,0.15
Stephon Castle,11.0,160.0,2594.56,0.91,0.76,0.75,0.15,0.16
Payton Pritchard,12.0,136.0,7604.68,0.58,0.49,0.43,0.10,0.15
De'Aaron Fox,17.0,590.0,18531.58,0.59,0.50,0.47,0.09,0.12
Jalen Brunson,19.0,278.0,5764.45,0.47,0.43,0.41,0.05,0.06
Shai Gilgeous-Alexander,16.0,283.0,12489.00,0.50,0.57,0.59,-0.07,-0.09
Mikal Bridges,15.0,466.0,9618.09,0.40,0.48,0.47,-0.08,-0.07



──────────── 5+ rebounds ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
James Harden,14.0,324.0,14255.44,0.64,0.55,0.53,0.09,0.11
Donovan Mitchell,11.0,163.0,4048.98,0.45,0.48,0.48,-0.03,-0.03
Shai Gilgeous-Alexander,11.0,147.0,6011.73,0.18,0.43,0.41,-0.24,-0.23



──────────── 6+ rebounds ────────────


...


──────────── 13+ rebounds ────────────
No data available for given constraint.

──────────── 14+ rebounds ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Nikola Jokić,17.0,275.0,6280.99,0.71,0.49,0.50,0.21,0.21
Karl-Anthony Towns,16.0,240.0,5433.69,0.25,0.28,0.27,-0.03,-0.02
Victor Wembanyama,25.0,1630.0,54210.12,0.40,0.48,0.48,-0.08,-0.07



──────────── 15+ rebounds ────────────
No data available for given constraint.

──────────── 16+ rebounds ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Victor Wembanyama,14.0,833.0,13906.7,0.14,0.29,0.3,-0.15,-0.16



──────────── 18+ rebounds ────────────
No data available for given constraint.

──────────── 20+ rebounds ────────────
No data available for given constraint.


In [78]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every rebounds prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_rebounds = set(int(r['rebounds'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_rebounds_map = dict()
for n in n_rebounds:
    df = rebounds_df[rebounds_df['n_rebounds'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_rebounds'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_rebounds_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_rebounds_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_rebounds'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Rebounds Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ rebounds'
)

fig.show()

In [79]:
# Simulation Statistics: Sell Yes, by Rebounds Threshold

stats_rows = []
for n, data in n_rebounds_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_rebounds': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_rebounds')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_rebounds'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_rebounds'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_rebounds'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Rebounds Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Rebounds', row=1, col=1)
fig.update_xaxes(title_text='n+ Rebounds', row=1, col=2)
fig.update_xaxes(title_text='n+ Rebounds', row=2, col=1)
fig.update_xaxes(title_text='n+ Rebounds', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()